# 04 - Task 3: Multi-Label Cuisine Classification

Predict cuisine labels (multi-label) from restaurant attributes with reproducible evaluation.

Leakage policy is defined in Notebook 01 and enforced via assertions in this notebook.

In [6]:
# Shared setup
import warnings
import random
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('seed:', SEED)

pandas: 2.2.3
numpy: 2.0.2
seed: 42


In [7]:
DATA_PATH = 'Dataset .csv'
df_raw = pd.read_csv(DATA_PATH)

EXPECTED_COLUMNS = [
    'Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
    'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
    'Average Cost for two', 'Currency', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Switch to order menu',
    'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes'
]
missing = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
assert not missing, f'Missing required columns: {missing}'
print('Loaded shape:', df_raw.shape)

Loaded shape: (9551, 21)


In [8]:
UNIVERSAL_DROP_COLUMNS = [
    'Restaurant ID', 'Restaurant Name', 'Address', 'Locality', 'Locality Verbose',
    'Longitude', 'Latitude', 'Switch to order menu',
    'Rating color', 'Rating text'
 ]
YES_NO_COLUMNS = ['Has Table booking', 'Has Online delivery', 'Is delivering now']

def map_yes_no(series: pd.Series) -> pd.Series:
    mapping = {'Yes': 1, 'No': 0}
    return series.map(mapping).fillna(series)

def cuisine_tokens(value):
    if pd.isna(value):
        return []
    tokens = [x.strip() for x in str(value).split(',') if x.strip()]
    return tokens

def apply_shared_preprocessing(df: pd.DataFrame, task: str) -> pd.DataFrame:
    out = df.copy()

    for c in YES_NO_COLUMNS:
        if c in out.columns:
            out[c] = map_yes_no(out[c])

    if task in {'task1', 'task2'}:
        out['Cuisines'] = out['Cuisines'].fillna('Unknown')
    elif task == 'task3':
        out = out.dropna(subset=['Cuisines']).copy()

    out['Cuisine Tokens'] = out['Cuisines'].apply(cuisine_tokens)

    drop_cols = [c for c in UNIVERSAL_DROP_COLUMNS if c in out.columns]
    out = out.drop(columns=drop_cols)
    return out

In [ ]:
df = apply_shared_preprocessing(df_raw, task='task3')

# Target leakage assertions
forbidden_features = ['Rating color', 'Rating text']
df = df.drop(columns=forbidden_features, errors='ignore')
assert not any(c in df.columns for c in forbidden_features), f'Leakage columns present: {forbidden_features}'

# Build multi-label targets
label_lists = df['Cuisine Tokens']

# Sparse label handling: keep labels with minimum support
min_support = 20
all_labels = pd.Series([lab for labs in label_lists for lab in labs])
label_counts = all_labels.value_counts()
kept_labels = set(label_counts[label_counts >= min_support].index)

filtered_labels = label_lists.apply(lambda labs: [x for x in labs if x in kept_labels])
mask = filtered_labels.apply(len) > 0
df = df[mask].copy()
filtered_labels = filtered_labels[mask]

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(filtered_labels)
print('Rows after filtering:', df.shape[0])
print('Number of labels:', len(mlb.classes_))

Rows after filtering: 9421
Number of labels: 57


In [10]:
feature_cols = [
    'Country Code', 'City', 'Average Cost for two', 'Currency',
    'Has Table booking', 'Has Online delivery', 'Is delivering now',
    'Price range', 'Votes', 'Aggregate rating'
]
X = df[feature_cols].copy()

categorical_features = ['City', 'Currency']
numeric_features = [
    'Country Code', 'Average Cost for two', 'Has Table booking',
    'Has Online delivery', 'Is delivering now', 'Price range', 'Votes', 'Aggregate rating'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(with_mean=False), numeric_features)
    ]
)

# Reproducible split; iterative stratification not used due package availability constraints.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=SEED
)

In [11]:
models = {
    'OneVsRest_LogisticRegression': OneVsRestClassifier(
        LogisticRegression(max_iter=2000, class_weight='balanced')
    ),
    'MultiOutput_RandomForest': MultiOutputClassifier(
        RandomForestClassifier(n_estimators=250, random_state=SEED, n_jobs=-1, class_weight='balanced_subsample')
    )
}

trained = {}
summary = []
predictions = {}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, Y_train)
    pred = pipe.predict(X_test)

    p, r, f1, _ = precision_recall_fscore_support(
        Y_test, pred, average='weighted', zero_division=0
    )
    subset_acc = accuracy_score(Y_test, pred)

    summary.append({
        'Model': name,
        'Weighted Precision': p,
        'Weighted Recall': r,
        'Weighted F1': f1,
        'Subset Accuracy': subset_acc
    })

    trained[name] = pipe
    predictions[name] = pred

summary_df = pd.DataFrame(summary).sort_values('Weighted F1', ascending=False).reset_index(drop=True)
display(summary_df)

,Model,Weighted Precision,Weighted Recall,Weighted F1,Subset Accuracy
0,OneVsRest_LogisticRegression,0.258043,0.702731,0.342831,0.000000
1,MultiOutput_RandomForest,0.328114,0.307248,0.307945,0.062069


In [12]:
best_model_name = summary_df.loc[0, 'Model']
best_pred = predictions[best_model_name]

label_precision, label_recall, label_f1, label_support = precision_recall_fscore_support(
    Y_test, best_pred, average=None, zero_division=0
)

per_label_df = pd.DataFrame({
    'Label': mlb.classes_,
    'Precision': label_precision,
    'Recall': label_recall,
    'F1': label_f1,
    'Support': label_support
}).sort_values('F1', ascending=False)

display(per_label_df.head(20))
display(per_label_df.tail(20))

,Label,Precision,Recall,F1,Support
40,North Indian,0.529982,0.735618,0.616094,817
9,Brazilian,0.333333,1.000000,0.500000,6
13,Chinese,0.376704,0.570356,0.453731,533
17,Fast Food,0.278675,0.856397,0.420513,383
14,Continental,0.273196,0.731034,0.397749,145
48,Southern,0.241379,0.875000,0.378378,8
21,Grill,0.217391,0.625000,0.322581,8
27,Italian,0.192817,0.723404,0.304478,141
45,Sandwich,0.183333,0.687500,0.289474,16
25,Indian,0.175676,0.812500,0.288889,16


,Label,Precision,Recall,F1,Support
3,BBQ,0.035714,0.285714,0.063492,7
8,Biryani,0.031100,0.764706,0.059770,34
19,French,0.029630,0.666667,0.056738,6
44,Salad,0.029630,0.631579,0.056604,19
43,Raw Meats,0.027704,0.875000,0.053708,24
20,Goan,0.022857,0.666667,0.044199,6
33,Lebanese,0.021692,0.555556,0.041754,18
37,Middle Eastern,0.020690,0.600000,0.040000,5
34,Malaysian,0.020202,1.000000,0.039604,4
52,Tea,0.010050,0.571429,0.019753,7


In [13]:
# Confusion trends by label: aggregate false positives and false negatives
fp = ((Y_test == 0) & (best_pred == 1)).sum(axis=0)
fn = ((Y_test == 1) & (best_pred == 0)).sum(axis=0)
tp = ((Y_test == 1) & (best_pred == 1)).sum(axis=0)

trend_df = pd.DataFrame({
    'Label': mlb.classes_,
    'TP': tp,
    'FP': fp,
    'FN': fn
}).sort_values(['FN', 'FP'], ascending=False)

display(trend_df.head(20))

,Label,TP,FP,FN
13,Chinese,304,503,229
40,North Indian,601,533,216
39,Mughlai,114,590,74
17,Fast Food,328,849,55
15,Desserts,90,648,48
12,Cafe,92,566,46
4,Bakery,111,843,44
47,South Indian,68,868,43
27,Italian,102,427,39
14,Continental,106,282,39


In [14]:
weak_labels = per_label_df[per_label_df['Precision'] < 0.5].sort_values('Precision')
print('Labels with precision < 0.5:', len(weak_labels))
display(weak_labels)

if len(weak_labels) > 0:
    print('Concrete improvement: increase label quality by grouping semantically similar rare cuisines and adding text features from Locality/Address embeddings.')
else:
    print('No weak labels under the 0.5 precision threshold.')

Labels with precision < 0.5: 56


,Label,Precision,Recall,F1,Support
32,Korean,0.000000,0.000000,0.000000,2
23,Hyderabadi,0.002188,0.500000,0.004357,2
54,Tibetan,0.004224,0.666667,0.008395,6
42,Rajasthani,0.004608,0.400000,0.009112,5
29,Juices,0.004662,0.500000,0.009238,4
6,Bengali,0.005891,0.666667,0.011679,6
30,Kashmiri,0.006757,0.500000,0.013333,6
56,Vietnamese,0.007092,0.250000,0.013793,4
31,Kerala,0.009416,0.714286,0.018587,7
1,Arabian,0.009740,0.300000,0.018868,10


Concrete improvement: increase label quality by grouping semantically similar rare cuisines and adding text features from Locality/Address embeddings.
